# 2— Preprocessing, Augmentation & EfficientNet-B0 Transfer Learning

Requirements covered:
- preprocessing
- augmentation
- class imbalance
- architecture selection
- transfer learning
- training


In [ ]:
from pathlib import Path
import json, random, time
import numpy as np, pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import matplotlib.pyplot as plt

SEED=42; BATCH=16; EPOCHS=10; LR=1e-3
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

ROOT=Path.cwd(); DATA=ROOT/"data"; MODELS=ROOT/"models"; RESULTS=ROOT/"results"
MODELS.mkdir(exist_ok=True); RESULTS.mkdir(exist_ok=True)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:",DEVICE)


In [ ]:
train_tf=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=.15,contrast=.15),
    transforms.ToTensor(),
    transforms.Normalize([.485,.456,.406],[.229,.224,.225])
])
eval_tf=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([.485,.456,.406],[.229,.224,.225])
])

train_ds=datasets.ImageFolder(DATA/"train",transform=train_tf)
val_ds=datasets.ImageFolder(DATA/"val",transform=eval_tf)
train_loader=DataLoader(train_ds,batch_size=BATCH,shuffle=True,num_workers=0)
val_loader=DataLoader(val_ds,batch_size=BATCH,shuffle=False,num_workers=0)
print("Classes:",train_ds.classes)
print("Mapping:",train_ds.class_to_idx)


In [ ]:
targets=torch.tensor(train_ds.targets)
counts=torch.bincount(targets,minlength=len(train_ds.classes)).float()
class_weights=counts.sum()/(len(counts)*counts)

print("Counts:",dict(zip(train_ds.classes,counts.int().tolist())))
print("Weights:",dict(zip(train_ds.classes,class_weights.tolist())))


In [ ]:
# EfficientNet-B0 with ImageNet transfer learning.
model=efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)

for p in model.features.parameters():
    p.requires_grad=False

model.classifier[1]=nn.Linear(model.classifier[1].in_features,len(train_ds.classes))
model=model.to(DEVICE)

criterion=nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
optimizer=torch.optim.AdamW(
    filter(lambda p:p.requires_grad,model.parameters()),
    lr=LR,weight_decay=1e-4
)

print(model.classifier)


In [ ]:
def validate():
    model.eval(); loss_sum=correct=total=0
    with torch.no_grad():
        for x,y in val_loader:
            x,y=x.to(DEVICE),y.to(DEVICE)
            out=model(x); loss=criterion(out,y)
            loss_sum+=loss.item()*y.size(0)
            correct+=(out.argmax(1)==y).sum().item()
            total+=y.size(0)
    return loss_sum/total, correct/total

history=[]; best=float("inf")
for epoch in range(EPOCHS):
    model.train(); loss_sum=correct=total=0
    for x,y in train_loader:
        x,y=x.to(DEVICE),y.to(DEVICE)
        optimizer.zero_grad()
        out=model(x); loss=criterion(out,y)
        loss.backward(); optimizer.step()
        loss_sum+=loss.item()*y.size(0)
        correct+=(out.argmax(1)==y).sum().item()
        total+=y.size(0)

    tr_loss=loss_sum/total; tr_acc=correct/total
    va_loss,va_acc=validate()
    history.append({"epoch":epoch+1,"train_loss":tr_loss,"train_accuracy":tr_acc,"val_loss":va_loss,"val_accuracy":va_acc})
    print(f"Epoch {epoch+1}/{EPOCHS} | train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} val_loss={va_loss:.4f} val_acc={va_acc:.4f}")

    if va_loss<best:
        best=va_loss
        torch.save(model.state_dict(),MODELS/"best_model.pth")
        (MODELS/"class_names.json").write_text(json.dumps(train_ds.classes,indent=2))

(RESULTS/"training_history.json").write_text(json.dumps(history,indent=2))


In [ ]:
hist=pd.DataFrame(history)
hist.plot(x="epoch",y=["train_loss","val_loss"],title="Training vs Validation Loss")
plt.tight_layout(); plt.show()
print("Saved:",MODELS/"best_model.pth")
